# Phase 1.1: Full TPU Sweep
## Complete normalization × D × η grid scan

**Protocol:** 4 norm types × 6 D × 8 η × 5 seeds, 40 epochs

This notebook performs the full sweep over all normalization types (BatchNorm, LayerNorm, GroupNorm, None) across multiple channel widths (D) and learning rates (η).

**What to expect:**
- Comprehensive data for β vs ln(γ) analysis across all norm types
- D-scaling fits for each norm type
- F-test for equal slopes across all normalization types

In [ ]:
# Cell 1: Install dependencies
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'numpy', 'scipy', 'matplotlib', 'pyyaml', 'tqdm'], check=True)
print("Dependencies installed.")

In [ ]:
# Cell 2: Model definitions and utilities (self-contained)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit
from typing import Dict, List, Optional, Tuple
import json
import os
from datetime import datetime
from tqdm import tqdm

# =============================================================================
# MODEL DEFINITION
# =============================================================================

class ConvNetL5(nn.Module):
    """ConvNet with L=5 layers, configurable normalization."""
    
    def __init__(self, D: int = 64, num_classes: int = 10, norm_type: str = 'batchnorm',
                 group_size: int = 4, input_channels: int = 3, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        self.D = D
        self.num_classes = num_classes
        self.norm_type = norm_type
        self.L = 5
        self.group_size = group_size
        
        self.conv1 = nn.Conv2d(input_channels, D, kernel_size, padding=padding)
        self.norm1 = self._make_norm(D)
        self.conv2 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm2 = self._make_norm(D)
        self.conv3 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm3 = self._make_norm(D)
        self.conv4 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm4 = self._make_norm(D)
        self.conv5 = nn.Conv2d(D, D, kernel_size, padding=padding)
        self.norm5 = self._make_norm(D)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(D, num_classes)
        self.activation = nn.GELU()
        self._initialize_weights()
        self._stored_activations = {}
    
    def _make_norm(self, num_channels: int) -> Optional[nn.Module]:
        if self.norm_type == 'batchnorm':
            return nn.BatchNorm2d(num_channels, affine=False)
        elif self.norm_type == 'layernorm':
            return nn.GroupNorm(1, num_channels, affine=False)
        elif self.norm_type == 'groupnorm':
            return nn.GroupNorm(self.group_size, num_channels, affine=False)
        elif self.norm_type == 'none':
            return None
        else:
            raise ValueError(f"Unknown norm_type: {self.norm_type}")
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                if m.weight is not None:
                    nn.init.ones_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    
    def _get_norm_output(self, x: torch.Tensor, norm_layer) -> torch.Tensor:
        if norm_layer is None:
            return x
        if isinstance(norm_layer, nn.BatchNorm2d):
            if self.training:
                return norm_layer(x)
            else:
                return (x - norm_layer.running_mean.view(1, -1, 1, 1)) / \
                       torch.sqrt(norm_layer.running_var.view(1, -1, 1, 1) + norm_layer.eps)
        elif isinstance(norm_layer, (nn.LayerNorm, nn.GroupNorm)):
            return norm_layer(x)
        return x
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.activation(self._get_norm_output(x, self.norm1))
        x = self.conv2(x)
        x = self.activation(self._get_norm_output(x, self.norm2))
        x = self.conv3(x)
        x = self.activation(self._get_norm_output(x, self.norm3))
        x = self.conv4(x)
        x = self.activation(self._get_norm_output(x, self.norm4))
        x = self.conv5(x)
        x = self.activation(self._get_norm_output(x, self.norm5))
        x = self.global_pool(x).flatten(1)
        return self.fc(x)
    
    def get_all_weights(self) -> List[torch.Tensor]:
        return [getattr(self, f'conv{i}').weight.data for i in range(1, 6)]

def create_model(D: int, norm_type: str, **kwargs) -> ConvNetL5:
    return ConvNetL5(D=D, norm_type=norm_type, **kwargs)

# =============================================================================
# ACTIVATION CAPTURE
# =============================================================================

def capture_activations(model, dataloader, device):
    """Capture normalized activations per layer."""
    model.eval()
    activations = []
    
    for data, _ in dataloader:
        data = data.to(device)
        x = data
        for i in range(1, 6):
            conv = getattr(model, f'conv{i}')
            norm = getattr(model, f'norm{i}')
            x = conv(x)
            if norm is not None:
                if isinstance(norm, nn.BatchNorm2d):
                    x_norm = (x - norm.running_mean.view(1, -1, 1, 1)) / \
                             torch.sqrt(norm.running_var.view(1, -1, 1, 1) + norm.eps)
                else:
                    x_norm = norm(x)
            else:
                x_norm = x
            activations.append(x_norm.cpu().detach())
            x = model.activation(x_norm if norm is not None else x)
        break
    return activations

def compute_sigma(activations: List[torch.Tensor]) -> np.ndarray:
    """Compute ℓ₂ norm per layer from activations."""
    sigmas = []
    for act in activations:
        act_flat = act.flatten(start_dim=1)
        l2_per_sample = act_flat.norm(p=2, dim=1)
        mean_l2 = l2_per_sample.mean().item()
        sigmas.append(mean_l2)
    return np.array(sigmas)


def compute_gamma_init(sigma_init: np.ndarray, norm_type: str, L: int = 5) -> Optional[float]:
    """Compute γ_init = (1/L) * Σ |ln(σ_ref / σ_init)| where σ_ref = 1 for BN/LN/GN."""
    if norm_type in ('batchnorm', 'layernorm', 'groupnorm'):
        sigma_ref = 1.0
        log_ratios = np.abs(np.log(sigma_ref / sigma_init))
        return float(np.mean(log_ratios))
    return None

# =============================================================================
# λ_max MEASUREMENT
# =============================================================================

def power_iteration_single_layer(W: torch.Tensor, num_iterations: int = 20, tol: float = 1e-6) -> float:
    """Compute λ_max via power iteration."""
    if W.dim() == 4:
        W_mat = W.reshape(W.shape[0], -1)
    else:
        W_mat = W
    
    if W_mat.shape[0] != W_mat.shape[1]:
        M = W_mat.T @ W_mat
    else:
        M = W_mat
    d = M.shape[0]
    
    torch.manual_seed(42)
    b = torch.randn(d)
    b = b / b.norm()
    
    lambda_prev = 0.0
    for _ in range(num_iterations):
        Mb = M @ b
        lambda_curr = Mb.norm().item()
        b = Mb / lambda_curr
        if abs(lambda_curr - lambda_prev) < tol:
            break
        lambda_prev = lambda_curr
    
    return torch.sqrt(torch.tensor(lambda_curr)).item()

def measure_lambda_max_mean(model: nn.Module, device: torch.device, num_iterations: int = 20) -> float:
    model.eval()
    lambda_values = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            W = module.weight.data.to(device)
            lambda_values.append(power_iteration_single_layer(W, num_iterations))
    return np.mean(lambda_values)

# =============================================================================
# STATIONARITY CHECK
# =============================================================================

def compute_loss_monotonicity_ratio(losses: np.ndarray, window_size: int = 20) -> float:
    if len(losses) < window_size:
        return 0.0
    window = losses[-window_size:]
    deltas = np.diff(window)
    if len(deltas) < 2:
        return 1.0
    signs = np.sign(deltas)
    sign_changes = np.sum(signs[:-1] != signs[1:])
    same_sign_count = len(signs) - 1 - sign_changes
    return same_sign_count / (len(signs) - 1)

def is_stationary(loss_history: np.ndarray, window_size: int = 20, monotonicity_threshold: float = 0.85) -> Tuple[bool, int]:
    if len(loss_history) < window_size:
        return False, -1
    for start_idx in range(len(loss_history) - window_size + 1):
        window = loss_history[start_idx:start_idx + window_size]
        mono_ratio = compute_loss_monotonicity_ratio(window, window_size)
        if mono_ratio >= monotonicity_threshold:
            return True, start_idx + window_size - 1
    return False, -1

# =============================================================================
# D-SCALING FIT
# =============================================================================

def d_scaling_model(D: np.ndarray, alpha: float, beta: float, E_floor: float) -> np.ndarray:
    return alpha * np.power(D, -beta) + E_floor

def fit_beta(losses: np.ndarray, D_values: np.ndarray, r2_threshold: float = 0.995) -> Tuple[Optional[float], Optional[float], float]:
    try:
        E_floor_guess = np.min(losses)
        alpha_guess = np.max(losses) - E_floor_guess
        beta_guess = 0.5
        
        popt, _ = curve_fit(d_scaling_model, D_values, losses, p0=[alpha_guess, beta_guess, E_floor_guess],
                           bounds=([0, 0, 0], [np.inf, 5, np.max(losses)]), maxfev=10000)
        alpha, beta, E_floor = popt
        
        y_pred = d_scaling_model(D_values, alpha, beta, E_floor)
        ss_res = np.sum((losses - y_pred) ** 2)
        ss_tot = np.sum((losses - np.mean(losses)) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        if r_squared >= r2_threshold:
            return beta, alpha, r_squared
        else:
            return None, None, r_squared
    except Exception as e:
        return None, None, 0.0

# =============================================================================
# TRAINING UTILITIES
# =============================================================================



def ancova_test_equal_slopes(raw_data: List[Dict], alpha: float = 0.05) -> Dict:
    """ANCOVA test for equal slopes across normalization types.
    H0: All interaction terms α_j = 0 (equal slopes)
    H1: At least one α_j ≠ 0 (slopes differ)"""
    try:
        import pandas as pd
        import statsmodels.api as sm
        import statsmodels.formula.api as smf
    except ImportError:
        return {'beta_spec': None, 'interaction_pvalue': None, 'reject_null': None,
                'message': 'statsmodels required for ANCOVA', 'fallback_to_f_test': True}
    df = pd.DataFrame(raw_data)
    df = df[df['gamma'] > 0].copy()
    df['ln_gamma'] = np.log(df['gamma'])
    if len(df) < 10:
        return {'beta_spec': None, 'interaction_pvalue': None, 'reject_null': None,
                'message': f'Insufficient data ({len(df)})', 'n_points': len(df)}
    norm_types = df['norm_type'].unique()
    if len(norm_types) < 2:
        return {'beta_spec': None, 'interaction_pvalue': None, 'reject_null': None,
                'message': 'Need 2+ norm types for ANCOVA', 'n_points': len(df)}
    formula = 'beta ~ C(norm_type) * np.log(gamma)'
    try:
        model = smf.ols(formula, data=df).fit()
    except Exception as e:
        return {'beta_spec': None, 'interaction_pvalue': None, 'reject_null': None,
                'message': f'ANCOVA model failed: {e}', 'fallback_to_f_test': True}
    try:
        anova_table = sm.stats.anova_lm(model, typ=3)
    except Exception as e:
        return {'beta_spec': None, 'interaction_pvalue': None, 'reject_null': None,
                'message': f'Type III ANOVA failed: {e}', 'fallback_to_f_test': True}
    interaction_label = 'C(norm_type):np.log(gamma)'
    if interaction_label not in anova_table.index:
        for idx in anova_table.index:
            if 'norm_type' in idx and 'ln_gamma' in idx:
                interaction_label = idx
                break
        else:
            return {'beta_spec': None, 'interaction_pvalue': None, 'reject_null': None,
                    'message': 'Interaction not found', 'fallback_to_f_test': True}
    interaction_pvalue = anova_table.loc[interaction_label, 'PR(>F)']
    beta_spec = model.params.get('np.log(gamma)')
    return {
        'beta_spec': float(beta_spec) if beta_spec is not None else None,
        'interaction_pvalue': float(interaction_pvalue) if interaction_pvalue is not None else None,
        'reject_null': bool(interaction_pvalue < alpha) if interaction_pvalue is not None else None,
        'r_squared': float(model.rsquared),
        'r_squared_adj': float(model.rsquared_adj),
        'n_points': int(len(df)),
        'n_norm_types': int(len(norm_types)),
        'model_summary': {'nobs': int(model.nobs), 'df_model': int(model.df_model), 'df_resid': int(model.df_resid)},
        'message': 'ANCOVA completed successfully',
        'fallback_to_f_test': False
    }

def train_single_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    for data, target in dataloader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    for data, target in dataloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = F.cross_entropy(output, target)
        total_loss += loss.item()
    return total_loss / len(dataloader)

def get_run_id(D: int, norm_type: str, lr: float, seed: int) -> str:
    lr_str = f"{lr:.6f}".rstrip('0')
    return f"norm_{norm_type}_D{D}_lr{lr_str}_seed{seed}"

def save_results(results: Dict, filepath: str):
    os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else '.', exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(results, f, indent=2)

def save_run_result(result: Dict, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    cfg = result['config']
    run_id = get_run_id(cfg.get('D'), cfg.get('norm_type'), cfg.get('lr'), cfg.get('seed'))
    filepath = os.path.join(output_dir, f"result_{run_id}.json")
    save_results(result, filepath)

def load_run_results(output_dir: str) -> List[Dict]:
    results = []
    if not os.path.exists(output_dir):
        return results
    for fname in os.listdir(output_dir):
        if fname.startswith('result_') and fname.endswith('.json'):
            with open(os.path.join(output_dir, fname), 'r') as f:
                results.append(json.load(f))
    return results

# =============================================================================
# DATA LOADING
# =============================================================================

def load_cifar10(batch_size: int = 128):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    data_dir = '/kaggle/input' if os.path.exists('/kaggle/input') else './data'
    
    trainset = torchvision.datasets.CIFAR10(root=data_dir, train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform_test)
    
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    return trainloader, testloader

print("Model and utilities defined.")

In [ ]:
# Cell 3: Configuration and Execution
OUTPUT_DIR = '/kaggle/working/phase_1.1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Phase 1.1 Configuration
NORM_TYPES = ['batchnorm', 'layernorm', 'groupnorm', 'none']
D_VALUES = [32, 64, 128, 256, 512, 1024]
LR_VALUES = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
SEEDS = [42, 43, 44, 45, 46]
EPOCHS = 40
BATCH_SIZE = 128

# Check for resume
existing_results = load_run_results(OUTPUT_DIR)
existing_run_ids = set()
for r in existing_results:
    cfg = r['config']
    existing_run_ids.add(get_run_id(cfg.get('D'), cfg.get('norm_type'), cfg.get('lr'), cfg.get('seed')))

print(f"Found {len(existing_run_ids)} existing results for resume")

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load CIFAR-10
trainloader, testloader = load_cifar10(batch_size=BATCH_SIZE)
print(f"Loaded CIFAR-10: {len(trainloader.dataset)} train, {len(testloader.dataset)} test samples")

total_runs = len(NORM_TYPES) * len(D_VALUES) * len(LR_VALUES) * len(SEEDS)
run_idx = 0

for norm_type in NORM_TYPES:
    for D in D_VALUES:
        for lr in LR_VALUES:
            for seed in SEEDS:
                run_idx += 1
                run_id = get_run_id(D, norm_type, lr, seed)
                
                if run_id in existing_run_ids:
                    print(f"[Skip {run_idx}/{total_runs}] {run_id} - already completed")
                    continue
                
                print(f"\n[{run_idx}/{total_runs}] norm={norm_type}, D={D}, lr={lr}, seed={seed}")
                
                torch.manual_seed(seed)
                np.random.seed(seed)
                
                model = create_model(D=D, norm_type=norm_type).to(device)
                optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
                
                # Initial measurements
                activations_init = capture_activations(model, trainloader, device)
                sigma_init = compute_sigma(activations_init)
                lambda_max_init = measure_lambda_max_mean(model, device, num_iterations=20)
                
                loss_history = []
                is_stationary_reached = False
                
                for epoch in tqdm(range(EPOCHS), desc=f"Epochs", leave=False):
                    train_loss = train_single_epoch(model, trainloader, optimizer, device)
                    eval_loss = evaluate(model, testloader, device)
                    loss_history.append(eval_loss)
                    
                    # Check stationarity
                    loss_array = np.array(loss_history)
                    is_stat, _ = is_stationary(loss_array)
                    if is_stat and not is_stationary_reached:
                        is_stationary_reached = True
                        print(f"  [Stationary] Reached at epoch {epoch}")
                    
                    scheduler.step()
                
                # Final measurements
                activations_final = capture_activations(model, trainloader, device)
                sigma_final = compute_sigma(activations_final)
                lambda_max_final = measure_lambda_max_mean(model, device, num_iterations=20)
                
                # Compute γ
                gamma = np.mean(np.abs(np.log(sigma_final / sigma_init)))
                
                result = {
                    'config': {'D': D, 'norm_type': norm_type, 'lr': lr, 'seed': seed, 'num_epochs': EPOCHS},
                    'sigma_init': sigma_init.tolist(),
                    'sigma_final': sigma_final.tolist(),
                    'gamma': float(gamma),
                    'gamma_init': gamma_init,
                    'lambda_max_init': float(lambda_max_init),
                    'lambda_max_final': float(lambda_max_final),
                    'loss_history': [float(l) for l in loss_history],
                    'stationary': is_stationary_reached,
                    'is_converged': loss_history[-1] < loss_history[0] * 0.7,
                    'completed': True,
                }
                
                save_run_result(result, OUTPUT_DIR)
                existing_run_ids.add(run_id)

print(f"\nTraining complete.")

In [ ]:
# Cell 4: Results Analysis and Saving
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("ANALYSIS: FULL SWEEP RESULTS")
print("="*70)

# Load all results
all_results = load_run_results(OUTPUT_DIR)
print(f"Loaded {len(all_results)} run results")

# Aggregate and fit β vs ln(γ) for each norm type
regression_results = {}
NORM_COLORS = {'batchnorm': '#E63946', 'layernorm': '#2A9D8F', 'groupnorm': '#E9C46A', 'none': '#264653'}

for norm_type in NORM_TYPES:
    beta_gamma_pairs = []
    
    for lr in LR_VALUES:
        losses_for_lr = []
        gammas_for_lr = []
        
        for D in D_VALUES:
            for seed in SEEDS:
                run_id = get_run_id(D, norm_type, lr, seed)
                result_path = os.path.join(OUTPUT_DIR, f"result_{run_id}.json")
                if os.path.exists(result_path):
                    with open(result_path, 'r') as f:
                        r = json.load(f)
                        losses_for_lr.append((D, r['loss_history'][-1]))
                        gammas_for_lr.append((D, r['gamma']))
        
        if len(losses_for_lr) >= 3:
            D_arr = np.array([x[0] for x in losses_for_lr])
            loss_arr = np.array([x[1] for x in losses_for_lr])
            beta, _, _ = fit_beta(loss_arr, D_arr)
            gamma_avg = np.mean([x[1] for x in gammas_for_lr])
            
            if beta is not None:
                beta_gamma_pairs.append((beta, gamma_avg))
    
    if len(beta_gamma_pairs) >= 3:
        betas = np.array([p[0] for p in beta_gamma_pairs])
        gammas = np.array([p[1] for p in beta_gamma_pairs])
        valid_mask = gammas > 0
        ln_gamma = np.log(gammas[valid_mask])
        betas_valid = betas[valid_mask]
        
        slope, intercept, r_value, p_value, std_err = stats.linregress(ln_gamma, betas_valid)
        
        regression_results[norm_type] = {
            'm': slope,
            'c': intercept,
            'r_squared': r_value ** 2,
            'p_value': p_value,
            'std_err': std_err,
            'n_points': len(beta_gamma_pairs),
        }
        
        print(f"\n{norm_type}:")
        print(f"  Slope m = {slope:.4f}")
        print(f"  Intercept c = {intercept:.4f}")
        print(f"  R² = {r_value**2:.4f}")

# Generate β vs ln(γ) plot
fig, ax = plt.subplots(figsize=(10, 7))

for norm_type, res in regression_results.items():
    if res['m'] is not None:
        x_range = np.linspace(-3, 1, 100)
        y_line = res['m'] * x_range + res['c']
        ax.plot(x_range, y_line, color=NORM_COLORS[norm_type], linewidth=2, 
                label=f"{norm_type}: m={res['m']:.3f}, R²={res['r_squared']:.3f}")

ax.set_xlabel('ln(γ)', fontsize=12)
ax.set_ylabel('β', fontsize=12)
ax.set_title('β vs ln(γ) - Full Sweep', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'beta_vs_ln_gamma.png'), dpi=150)
plt.show()

# Generate D-scaling collapse plot
fig, ax = plt.subplots(figsize=(10, 7))

for norm_type in NORM_TYPES:
    for lr in LR_VALUES[:4]:  # Plot subset for clarity
        D_losses = []
        for D in D_VALUES:
            for seed in SEEDS:
                run_id = get_run_id(D, norm_type, lr, seed)
                result_path = os.path.join(OUTPUT_DIR, f"result_{run_id}.json")
                if os.path.exists(result_path):
                    with open(result_path, 'r') as f:
                        D_losses.append((D, json.load(f)['loss_history'][-1]))
        
        if D_losses:
            unique_D = sorted(set([x[0] for x in D_losses]))
            avg_losses = []
            for D in unique_D:
                losses_at_D = [l for d, l in D_losses if d == D]
                avg_losses.append(np.mean(losses_at_D))
            ax.scatter(unique_D, avg_losses, color=NORM_COLORS[norm_type], alpha=0.6, s=60)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('D (channel width)', fontsize=12)
ax.set_ylabel('Final Loss ℒ', fontsize=12)
ax.set_title('D-Scaling Collapse', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'd_scaling_collapse.png'), dpi=150)
plt.show()


# ANCOVA test for equal slopes across all norm types
print("\n" + "="*70)
print("ANCOVA TEST FOR EQUAL SLOPES")
print("="*70)

raw_data_for_ancova = []
# Collect raw data points for ANCOVA
for norm_type in NORM_TYPES:
    for lr in LR_VALUES:
        for D in D_VALUES:
            for seed in SEEDS:
                run_id = get_run_id(D, norm_type, lr, seed)
                result_path = os.path.join(OUTPUT_DIR, f"result_{run_id}.json")
                if os.path.exists(result_path):
                    with open(result_path, 'r') as f:
                        r = json.load(f)
                        if 'loss_history' in r and len(r['loss_history']) > 0:
                            raw_data_for_ancova.append({
                                'beta': r.get('beta'),
                                'gamma': r.get('gamma'),
                                'norm_type': norm_type,
                                'D': D,
                                'lr': lr,
                                'seed': seed,
                                'gamma_init': r.get('gamma_init'),
                            })

ancova_result = ancova_test_equal_slopes(raw_data_for_ancova, alpha=0.05)
print(f"\nANCOVA Result:")
print(f"  β_spec (shared slope): {ancova_result.get('beta_spec')}")
print(f"  Interaction p-value: {ancova_result.get('interaction_pvalue')}")
if ancova_result.get('reject_null') is not None:
    print(f"  Slopes equal: {not ancova_result.get('reject_null')}")
print(f"  R²: {ancova_result.get('r_squared')}")
print(f"  Message: {ancova_result.get('message')}")

# Save final results
save_path = os.path.join(OUTPUT_DIR, 'phase_1.1_results.json')
save_results({'regression': regression_results, 'ancova': ancova_result, 'config': {
    'norm_types': NORM_TYPES,
    'D_values': D_VALUES,
    'lr_values': LR_VALUES,
    'seeds': SEEDS,
    'epochs': EPOCHS
}}, save_path)
print(f"\nResults saved to {save_path}")